# Keren 2018 (TNBC, MIBI-TOF): cell table from the images and the published masks

**Why.** Keren measured **H3K27me3** (clone C36B11, the same as Jackson) and **H3K9ac**. Danenberg measured neither mark, so
Keren is the only cohort outside Jackson where an imputed H3K27me3 can be checked against the measured mark cell by cell
(`epi-relations`). Keren has no pHH3.

**Input:** Kaggle dataset `yuvalsamoilov/keren2018`
- `1.tiff`..`41.tiff`: 36 channels, raw ion counts, 2048 × 2048 px over 800 µm (0.39 µm/px). The stacks carry no channel
  names; the order is assumed to be `panel.parquet` and is **checked here** against the published cell table.
- `p#_labeledcellData.tiff`: the published segmentation (labels 1..N; some objects are background, not cells).
- `cellData.csv`: the published cell table (counts / cell size → arcsinh → standardised; plus cell classes).
- `patient_class.csv`: tumour–immune architecture (0 mixed, 1 compartmentalised, 2 cold).

**What this kernel does:**
1. Per-cell **raw mean counts** for all 36 channels under the published masks, so Keren can be put on the same
   arcsinh(x / cofactor) scale as the IMC cohorts. The published values are standardised and cannot be.
2. The same means under masks **eroded by 2 µm** (5 px here; 2 px at the IMC 1 µm/px) for the spillover check.
3. **Channel-order check:** the within-image rank correlation between every published column and every image channel.
   The channel order is accepted only if the best one-to-one matching is the panel order for the two marks and the eight
   anchors (the inputs shared with Jackson and Danenberg).
4. Joins the published cell classes. Mask objects missing from the published table are background and are dropped.

**Output** (same schema as `jackson-cells-prep` where it applies): `cells.parquet`, `clinical.parquet`, `mask_stats.csv`,
`channel_check.csv`, `channel_scale.csv`, `prep_report.json`.

In [ ]:
import os, glob, json, re, time, sys, subprocess, warnings, traceback
import numpy as np, pandas as pd
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")
try:
    import imagecodecs  # noqa: F401
except ImportError:
    pip = [sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:"]
    if subprocess.run(pip + ["imagecodecs==2024.9.22"]).returncode:      # no wheel for this Python: newest binary
        subprocess.run(pip + ["imagecodecs"], check=True)
import tifffile

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
MAX_IMAGES = int(os.environ.get("MAX_IMAGES", 0))     # > 0: local smoke test on the first images only
PIXEL_UM = 800 / 2048        # Keren 2018: 800 µm fields at 2048 × 2048 px
ERODE_UM = 2.0               # the IMC spillover check erodes 2 px at 1 µm/px; same physical distance here
ERODE_PX = int(round(ERODE_UM / PIXEL_UM))
N_JOBS = min(3, os.cpu_count())                       # one image stack is 0.6 GB as float32
N_CHECK = 60_000
SEED = 20261022
EPI = ["h3k27me3", "h3k9ac"]
ANCHORS = ["ds_dna", "ki67", "cd3", "cd20", "cd45", "cd68", "sma", "pan_keratin"]   # shared with Jackson and Danenberg
GROUP = {1: "unidentified", 2: "immune", 3: "endothelial", 4: "mesenchymal", 5: "tumour", 6: "keratin_pos_tumour"}
BROAD = {1: "other", 2: "immune", 3: "stromal", 4: "stromal", 5: "epithelial", 6: "epithelial"}
IMMUNE = {1: "treg", 2: "cd4_t", 3: "cd8_t", 4: "cd3_t", 5: "nk", 6: "b", 7: "neutrophil", 8: "macrophage", 9: "dc",
          10: "dc_mono", 11: "mono_neu", 12: "other_immune"}
ARCH = {0: "mixed", 1: "compartmentalised", 2: "cold"}
T0 = time.time()
REPORT = dict(pixel_um=PIXEL_UM, erode_px=ERODE_PX, erode_um=ERODE_PX * PIXEL_UM)

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def one(pattern):
    f = g(pattern)
    assert len(f) == 1, (pattern, f)
    return f[0]

def jdump(d):
    return json.dumps(d, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))

def save_report():
    with open(f"{OUT}/prep_report.json", "w") as fh:
        fh.write(jdump(REPORT))

## Inputs

In [ ]:
panel = pd.read_parquet(one("panel.parquet"))
CH = panel.target.tolist()
TABLE_NAME = dict(zip(CH, panel.target_original_name))
assert len(CH) == 36 and all(c in CH for c in EPI + ANCHORS), CH
IMG, MASK = {}, {}
for p in g("*.tiff"):
    b = os.path.basename(p)
    if m := re.fullmatch(r"(\d+)\.tiff", b):
        IMG[int(m.group(1))] = p
    elif m := re.fullmatch(r"p(\d+)_labeledcellData\.tiff", b):
        MASK[int(m.group(1))] = p
tab = pd.read_csv(one("cellData.csv"))
assert all(TABLE_NAME[c] in tab.columns for c in CH), [TABLE_NAME[c] for c in CH if TABLE_NAME[c] not in tab.columns]
arch = pd.read_csv(one("patient_class.csv"), header=None, names=["sample", "arch"])
T_IDS = set(tab.SampleID.unique())
ids = sorted(set(IMG) & set(MASK) & T_IDS)
REPORT["inputs"] = dict(images=len(IMG), masks=len(MASK), table_samples=len(T_IDS), table_cells=len(tab), used=len(ids),
                        image_and_mask_but_no_table=sorted(set(IMG) & set(MASK) - T_IDS),
                        table_but_no_image=sorted(T_IDS - set(IMG)), image_without_mask=sorted(set(IMG) - set(MASK)))
if MAX_IMAGES:
    ids = ids[:MAX_IMAGES]
print(jdump(REPORT["inputs"]))
tick(f"{len(ids)} images to process")

## Per-cell means under the full and the eroded masks

In [ ]:
def erode_labels(lab, r):
    mx = ndimage.maximum_filter(lab, size=2 * r + 1, mode="constant", cval=0)
    mn = ndimage.minimum_filter(lab, size=2 * r + 1, mode="constant", cval=0)
    return np.where((mx == lab) & (mn == lab), lab, 0)

def per_image(s):
    im = tifffile.imread(IMG[s])
    lab = tifffile.imread(MASK[s]).astype(np.int64)
    assert im.shape == (len(CH),) + lab.shape, (s, im.shape, lab.shape)
    n = int(lab.max()) + 1
    flat, er = lab.ravel(), erode_labels(lab, ERODE_PX).ravel()
    cnt, cnt_e = np.bincount(flat, minlength=n).astype(np.float64), np.bincount(er, minlength=n).astype(np.float64)
    yy, xx = np.indices(lab.shape)
    out = dict(object_number=np.arange(n), area_px=cnt, area_px_eroded=cnt_e,
               y=np.bincount(flat, yy.ravel(), n) / np.maximum(cnt, 1), x=np.bincount(flat, xx.ravel(), n) / np.maximum(cnt, 1))
    st = dict(sample=s, height=lab.shape[0], width=lab.shape[1], n_objects=int((cnt[1:] > 0).sum()),
              background_px_frac=float((flat == 0).mean()))
    for j, ch in enumerate(CH):
        v = im[j].ravel().astype(np.float64)
        out[ch] = np.bincount(flat, v, n) / np.maximum(cnt, 1)
        out[f"eroded__{ch}"] = np.where(cnt_e > 0, np.bincount(er, v, n) / np.maximum(cnt_e, 1), np.nan)
        st[f"zero_px__{ch}"] = float((v == 0).mean())
        st[f"integer_px__{ch}"] = bool(np.all(v == np.round(v)))
    df = pd.DataFrame(out).iloc[1:]
    df = df[df.area_px > 0].copy()
    df.insert(0, "sample", s)
    return df, st

res = Parallel(n_jobs=N_JOBS)(delayed(per_image)(s) for s in ids)
obj = pd.concat([r[0] for r in res], ignore_index=True)
mstats = pd.DataFrame([r[1] for r in res]).set_index("sample")
del res
tick(f"{len(obj):,} mask objects in {len(ids)} images")

## Join the published table (cell classes); mask objects not in the table are background

In [ ]:
t = tab[tab.SampleID.isin(ids)].rename(columns={"SampleID": "sample", "cellLabelInImage": "object_number"})
dup = int(t.duplicated(["sample", "object_number"]).sum())
assert dup == 0, f"{dup} duplicated (sample, label) rows in the published table"
keep = ["sample", "object_number", "cellSize", "tumorYN", "Group", "immuneGroup"] + [TABLE_NAME[c] for c in CH]
t = t[keep].rename(columns={TABLE_NAME[c]: f"pub__{c}" for c in CH})   # one panel name (p53) equals its table name
M = obj.merge(t, on=["sample", "object_number"], how="outer", indicator=True)
n_tab_only = int((M._merge == "right_only").sum())
bg = M[M._merge == "left_only"].groupby("sample").size()
M = M[M._merge == "both"].drop(columns="_merge").reset_index(drop=True)
size_equal = float((M.cellSize == M.area_px).mean())
size_r = float(np.corrcoef(M.cellSize, M.area_px)[0, 1])
mstats["n_table_cells"] = M.groupby("sample").size()
mstats["n_background_objects"] = bg.reindex(mstats.index).fillna(0).astype(int)
mstats["table_cells_not_in_mask"] = t.groupby("sample").size().reindex(mstats.index) - mstats.n_table_cells
REPORT["join"] = dict(cells=len(M), table_cells_not_in_mask=n_tab_only, background_objects=int(bg.sum()),
                      cell_size_equal_frac=size_equal, cell_size_vs_area_r=size_r)
print(jdump(REPORT["join"]))
assert n_tab_only <= 0.01 * len(t), "many published cells have no mask object: masks and table do not match"
assert size_r > 0.95, "published cell size does not match the mask area"

## Channel-order check

Within each image, rank every published column and every image channel over the cells, then correlate (pooled over
images). If the stacks follow `panel.parquet`, each published column matches its own channel best.

In [ ]:
rs = np.random.default_rng(SEED)
sub = M.iloc[np.sort(rs.choice(len(M), min(N_CHECK, len(M)), replace=False))]
def within_rank(df, cols):
    r = df.groupby("sample")[cols].rank(pct=True)
    r = (r - r.groupby(df["sample"]).transform("mean")).values
    return r / np.sqrt((r ** 2).sum(0, keepdims=True))
RT = within_rank(sub, [f"pub__{c}" for c in CH])
RI = within_rank(sub, CH)
R = np.nan_to_num(RT.T @ RI)                     # rows: published column, columns: image channel
_, match = linear_sum_assignment(-R)
chk = pd.DataFrame({"channel": CH, "table_column": [TABLE_NAME[c] for c in CH], "r_same_channel": np.diag(R),
                    "best_image_channel": [CH[j] for j in R.argmax(1)], "r_best": R.max(1),
                    "matched_channel": [CH[j] for j in match]})
chk["ok"] = chk.channel == chk.matched_channel
chk.to_csv(f"{OUT}/channel_check.csv", index=False)
print(chk.round(3).to_string(index=False))
REPORT["channel_check"] = dict(n_cells=len(sub), all_channels_ok=bool(chk.ok.all()),
                               mismatched=chk.loc[~chk.ok, "channel"].tolist(),
                               r_same_channel={c: float(r) for c, r in zip(chk.channel, chk.r_same_channel)
                                               if c in EPI + ANCHORS})
save_report()
need = chk.set_index("channel").loc[EPI + ANCHORS]
assert need.ok.all() and (need.r_same_channel > 0.8).all(), f"channel order not confirmed:\n{need}"
tick("channel order confirmed for the marks and the anchors")

## Value scale (raw mean counts per cell)

In [ ]:
q = [0.5, 0.9, 0.99, 0.999]
scale = pd.DataFrame({ch: {**{f"q{int(p * 1000) / 10:g}": float(np.quantile(M[ch], p)) for p in q},
                           "max": float(M[ch].max()), "mean": float(M[ch].mean()),
                           "frac_cells_zero": float((M[ch] == 0).mean()),
                           "frac_px_zero_mean": float(mstats[f"zero_px__{ch}"].mean()),
                           "integer_counts": bool(mstats[f"integer_px__{ch}"].all())} for ch in CH}).T
scale.index.name = "channel"
scale.to_csv(f"{OUT}/channel_scale.csv")
print(scale.loc[EPI + ANCHORS].round(3).to_string())
REPORT["scale_marks_anchors"] = scale.loc[EPI + ANCHORS].to_dict(orient="index")

## Write the outputs

In [ ]:
c = pd.DataFrame({
    "image_name": M["sample"].astype(str), "object_number": M.object_number.astype(np.int64),
    "patient_id": M["sample"].astype(str), "cohort": "keren", "is_tumour": 1,
    "area": M.area_px.astype(np.float32), "area_um2": (M.area_px * PIXEL_UM ** 2).astype(np.float32),
    "area_eroded": M.area_px_eroded.astype(np.float32),
    "x": M.x.astype(np.float32), "y": M.y.astype(np.float32),
    "x_um": (M.x * PIXEL_UM).astype(np.float32), "y_um": (M.y * PIXEL_UM).astype(np.float32),
    "group": M.Group.astype(int), "group_name": M.Group.map(GROUP), "broad_class": M.Group.map(BROAD),
    "immune_group": M.immuneGroup.astype(int), "immune_group_name": M.immuneGroup.map(IMMUNE).fillna(""),
    "tumor_yn": M.tumorYN.astype(int)})
c["cell_phenotype"] = np.where(c.immune_group_name != "", c.immune_group_name, c.group_name)
for ch in CH:
    c[ch] = M[ch].astype(np.float32)
for ch in CH:
    c[f"eroded__{ch}"] = M[f"eroded__{ch}"].astype(np.float32)
for ch in EPI + ANCHORS:                            # the published (standardised) values, for traceability
    c[f"published__{ch}"] = M[f"pub__{ch}"].astype(np.float32)
c = c.sort_values(["patient_id", "object_number"], key=lambda s: s.astype(int) if s.name == "patient_id" else s)
c.to_parquet(f"{OUT}/cells.parquet", index=False)

clin = arch[arch["sample"].isin(ids)].assign(patient_id=lambda d: d["sample"].astype(str),
                                             immune_architecture=lambda d: d.arch.map(ARCH))
clin = clin.rename(columns={"arch": "immune_architecture_code"})[["patient_id", "immune_architecture_code",
                                                                  "immune_architecture"]]
clin["cohort"], clin["subtype"] = "keren", "TNBC"
clin.to_parquet(f"{OUT}/clinical.parquet", index=False)
mstats.drop(columns=[k for k in mstats if k.startswith("integer_px__")]).to_csv(f"{OUT}/mask_stats.csv")

REPORT["output"] = dict(cells=len(c), images=c.image_name.nunique(), patients=c.patient_id.nunique(),
                        broad_class=c.broad_class.value_counts().to_dict(),
                        cells_per_image=dict(c.groupby("image_name").size().describe().round(1)),
                        median_area_px=float(c.area.median()), median_area_um2=float(c.area_um2.median()),
                        median_area_eroded_px=float(c.area_eroded.median()),
                        cells_lost_by_erosion_frac=float((c.area_eroded == 0).mean()),
                        immune_architecture=clin.immune_architecture.value_counts().to_dict(),
                        note="Values are raw mean ion counts per cell (not transformed). Survival is not included; "
                             "it is in the Keren 2018 supplement.")
REPORT["minutes"] = (time.time() - T0) / 60
save_report()
print(jdump(REPORT["output"]))
tick("done")